# Agent: With vs Without DSPy | With LLM call

Same question, same model (`gpt-4o-mini`) — measuring latency, token usage, and output structure.

| Dimension | Without DSPy | With DSPy |
|-----------|-------------|-----------|
| **Prompt** | Hand-written instruction string | Typed `Signature` → auto-generated structured prompt |
| **Output** | Flat text blob | `reasoning` + `answer` typed fields |
| **Reasoning** | Implicit (hidden inside LLM) | Explicit, captured in `reasoning` field |
| **Optimizable?** | No — manual rewriting only | Yes — `dspy.MIPROv2` / `BootstrapFewShot` |

In [62]:
# ── Install ───────────────────────────────────────────────────────────────────
%pip install -q openai-agents dspy openai

# ── Imports ───────────────────────────────────────────────────────────────────
import os, time
from agents import Agent, Runner
import dspy
OPENAI_API_KEY = "API_KEY_HERE"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
MODEL = "gpt-4o-mini"

In [63]:
# ═══════════════════════════════════════════════════════════════════
# RUN 1 — WITHOUT DSPy  (plain OpenAI Agents SDK)
# ═══════════════════════════════════════════════════════════════════

plain_agent = Agent(
    name="PlainAgent",
    instructions=(
        "You are a helpful AI assistant. "
        "Answer questions clearly and concisely, providing all relevant details."
    ),
    model=MODEL,
)

t0 = time.perf_counter()
plain_result = await Runner.run(plain_agent, TEST_Q)
plain_time = time.perf_counter() - t0

# ── Token usage ───────────────────────────────────────────────────────────────
try:
    plain_tokens = sum(
        r.usage.total_tokens
        for r in plain_result.raw_responses
        if r.usage
    )
except Exception:
    plain_tokens = "N/A"

plain_output = plain_result.final_output

print("WITHOUT DSPy")
print("=" * 60)
print(f"Time   : {plain_time:.2f}s")
print(f"Tokens : {plain_tokens}")
print(f"\nOutput :\n{plain_output}")

WITHOUT DSPy
Time   : 3.10s
Tokens : 145

Output :
I don't have real-time weather data. However, Chennai typically has a tropical climate, characterized by warm temperatures and high humidity.

For a typical day in Chennai, consider wearing:

- Lightweight, breathable clothing (cotton or linen)
- Light-colored clothes to reflect sunlight
- Sunglasses and a wide-brimmed hat for sun protection
- Comfortable sandals or shoes
- An umbrella or light rain jacket if rain is expected

Please check a reliable weather service for current conditions and specific recommendations.


## Run 2 — With DSPy (ChainOfThought) | With Agent call

DSPy automatically builds a structured prompt from the `QA` signature class.  
The model returns typed `reasoning` and `answer` fields instead of a flat text blob.

In [64]:
# ═══════════════════════════════════════════════════════════════════
# RUN 2 — WITH DSPy  (ChainOfThought exposed as a function_tool)
#   Both runs go through Runner.run() — same Agents SDK loop.
#   The difference: Run 2's agent calls a DSPy tool for structured reasoning.
# ═══════════════════════════════════════════════════════════════════

from agents import function_tool

# ── 1. Configure DSPy LM (same underlying model) ─────────────────────────────
lm = dspy.LM(model=f"openai/{MODEL}", api_key=OPENAI_API_KEY)
dspy.configure(lm=lm)

# ── 2. Typed Signature — defines what the DSPy tool should produce ────────────
class QA(dspy.Signature):
    """You are a helpful AI assistant. Reason step-by-step and give a clear answer."""
    question: str  = dspy.InputField(desc="The user's question")
    reasoning: str = dspy.OutputField(desc="Step-by-step reasoning before answering")
    answer: str    = dspy.OutputField(desc="Clear, concise final answer with all relevant details")

cot = dspy.ChainOfThought(QA)

# ── 3. Wrap the DSPy module as a function_tool the agent can call ─────────────
_dspy_capture = {}   # stores the last prediction's structured fields

@function_tool
def think_with_dspy(question: str) -> str:
    """Use structured Chain-of-Thought reasoning to answer a question."""
    pred = cot(question=question)
    _dspy_capture["reasoning"] = pred.reasoning
    _dspy_capture["answer"]    = pred.answer
    return f"[Reasoning]\n{pred.reasoning}\n\n[Answer]\n{pred.answer}"

# ── 4. Agent that uses DSPy as its reasoning engine ───────────────────────────
dspy_agent = Agent(
    name="DSPyAgent",
    instructions=(
        "You are a helpful AI assistant. "
        "For every user question, call the think_with_dspy tool to reason through "
        "your answer, then return its response to the user."
    ),
    tools=[think_with_dspy],
    model=MODEL,
)

t0 = time.perf_counter()
dspy_run_result = await Runner.run(dspy_agent, TEST_Q)
dspy_time = time.perf_counter() - t0

# ── Token usage (DSPy tool call side) ────────────────────────────────────────
try:
    usage       = lm.history[-1].get("usage", {})
    dspy_tokens = usage.get("total_tokens") or (
        usage.get("prompt_tokens", 0) + usage.get("completion_tokens", 0)
    )
except Exception:
    dspy_tokens = "N/A"

dspy_final_output = dspy_run_result.final_output
dspy_reasoning    = _dspy_capture.get("reasoning", "")
dspy_answer       = _dspy_capture.get("answer", "")

print("WITH DSPy — ChainOfThought as function_tool inside Agents SDK")
print("=" * 60)
print(f"Time      : {dspy_time:.2f}s")
print(f"Tokens    : {dspy_tokens}")
print(f"\nDSPy tool — Reasoning :\n{dspy_reasoning}")
print(f"\nDSPy tool — Answer    :\n{dspy_answer}")
print(f"\nAgent final output    :\n{dspy_final_output}")

WITH DSPy — ChainOfThought as function_tool inside Agents SDK
Time      : 8.20s
Tokens    : 420

DSPy tool — Reasoning :
To provide an accurate response to the current weather in Chennai, I would typically rely on real-time weather data. However, I can offer general insights based on typical weather patterns for Chennai during the current season (which is usually hot and humid). 

Chennai, located in a tropical climate zone, generally experiences high temperatures throughout the year. In October, the weather begins to transition slightly due to the beginning of the northeast monsoon, so rain can occur, but it remains warm. 

Considering these factors, suitable clothing options for Chennai in October would be lightweight, breathable fabrics like cotton and linen to keep cool. Additionally, having an umbrella or a light rain jacket may be prudent for unexpected rains.

DSPy tool — Answer    :
The current weather in Chennai is typically warm and humid, with temperatures often ranging betw

## Side-by-Side Comparison

In [65]:
# ═══════════════════════════════════════════════════════════════════
# COMPARISON SUMMARY
# Both runs used the OpenAI Agents SDK (Runner.run).
# Run 2 additionally called a DSPy ChainOfThought tool for structured reasoning.
# ═══════════════════════════════════════════════════════════════════

rows = [
    ("Latency (s)",      f"{plain_time:.2f}",                    f"{dspy_time:.2f}"),
    ("DSPy tokens",      "N/A",                                  str(dspy_tokens)),
    ("Agent loop",       "Runner.run (plain)",                   "Runner.run + think_with_dspy tool"),
    ("Output fields",    "1  (flat text blob)",                  "reasoning + answer + agent summary"),
    ("Prompt source",    "Hand-written instructions string",     "Typed Signature → auto-generated"),
    ("Optimizable?",     "No",                                   "Yes (MIPROv2, BootstrapFewShot)"),
]

W = [20, 38, 38]
print(f"{'Metric':<{W[0]}} {'Without DSPy':<{W[1]}} {'With DSPy':<{W[2]}}")
print("-" * sum(W))
for label, a, b in rows:
    print(f"{label:<{W[0]}} {a:<{W[1]}} {b:<{W[2]}}")

print("\n── Output preview ──────────────────────────────────────────────────────")
print(f"\n[Without DSPy — Agent output]\n{plain_output[:400]}{'...' if len(plain_output) > 400 else ''}")
print(f"\n[With DSPy — Reasoning (from tool)]\n{dspy_reasoning[:400]}{'...' if len(dspy_reasoning) > 400 else ''}")
print(f"\n[With DSPy — Answer (from tool)]\n{dspy_answer[:400]}{'...' if len(dspy_answer) > 400 else ''}")
print(f"\n[With DSPy — Agent final output]\n{dspy_final_output[:400]}{'...' if len(dspy_final_output) > 400 else ''}")

Metric               Without DSPy                           With DSPy                             
------------------------------------------------------------------------------------------------
Latency (s)          3.10                                   8.20                                  
DSPy tokens          N/A                                    420                                   
Agent loop           Runner.run (plain)                     Runner.run + think_with_dspy tool     
Output fields        1  (flat text blob)                    reasoning + answer + agent summary    
Prompt source        Hand-written instructions string       Typed Signature → auto-generated      
Optimizable?         No                                     Yes (MIPROv2, BootstrapFewShot)       

── Output preview ──────────────────────────────────────────────────────

[Without DSPy — Agent output]
I don't have real-time weather data. However, Chennai typically has a tropical climate, characterized by 

# Prompting --> DSPy --> Agent

DSPy reasoning  →  becomes Agent instructions

     ↓
Agent now has an optimized thinking framework baked in

     ↓
Every subsequent question through this agent benefits from it

In [72]:
# %pip install -q openai-agents dspy openai

import os, time
import dspy
from agents import Agent, Runner

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "API_KEY_HERE")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
MODEL = "gpt-4o-mini"
TEST_Q = "What  Happens if we take coffee befor going bed?"


# ═══════════════════════════════════════════════════════════════════
# PHASE 1 — DSPy: Optimize the prompt (runs ONCE, offline)
#   DSPy takes your task description and automatically finds:
#     - the best system instructions
#     - the best reasoning style
#   Output: a concrete optimized prompt string
#   This is DSPy's entire job here. It touches no agent.
# ═══════════════════════════════════════════════════════════════════

lm = dspy.LM(model=f"openai/{MODEL}", api_key=OPENAI_API_KEY)
dspy.configure(lm=lm)

# Define the task structure for DSPy
class QA(dspy.Signature):
    """You are an good Q/A agent, helps the customers with the logical, reasoning and factual questions."""
    question: str = dspy.InputField()
    answer:   str = dspy.OutputField(desc="Clear, structured answer with examples where helpful")

# ChainOfThought: DSPy's way of building a reasoning-first prompt
cot = dspy.ChainOfThought(QA)

# --- Run DSPy on the question to get optimized output ---
print("=" * 60)
print("PHASE 1: DSPy generating optimized prompt + reasoning")
print("=" * 60)

dspy_start  = time.perf_counter()
dspy_pred   = cot(question=TEST_Q)
dspy_time   = time.perf_counter() - dspy_start

# Extract what DSPy produced
dspy_reasoning       = dspy_pred.reasoning   # step-by-step reasoning DSPy generated
dspy_suggested_answer = dspy_pred.answer     # DSPy's own answer attempt

# The key output: use DSPy's reasoning as enriched instructions for the Agent
# DSPy figured out HOW to think about this problem.
# We now hand that thinking style to the Agent as its system prompt.
optimized_instructions = f"""You are an good Q/A agent.

When answering, follow this reasoning approach that has been optimized for this type of question:

{dspy_reasoning}

Use the above reasoning as your thinking framework. Then give a clear, well-structured final answer with examples."""

# Token count from DSPy
dspy_usage  = lm.history[-1].get("usage", {})
dspy_tokens = dspy_usage.get("prompt_tokens", 0) + dspy_usage.get("completion_tokens", 0)

print(f"Latency : {dspy_time:.2f}s")
print(f"Tokens  : {dspy_tokens}")
print(f"\nDSPy Reasoning (will become Agent's instructions):\n{dspy_reasoning}")
print(f"\nDSPy's own answer attempt:\n{dspy_suggested_answer}")


# ═══════════════════════════════════════════════════════════════════
# PHASE 2 — Agents SDK: Execute with DSPy's optimized prompt
#
#   The Agent receives DSPy's optimized reasoning as its instructions.
#   It now knows exactly HOW to think about this class of problem.
#   One clean LLM call. No DSPy involved at runtime.
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print("PHASE 2: Agents SDK executing with DSPy-optimized prompt")
print("=" * 60)

agent = Agent(
    name="OptimizedAgent",
    instructions=optimized_instructions,   # ← DSPy's output becomes the agent's brain
    model=MODEL,
)

agent_start  = time.perf_counter()
agent_result = await Runner.run(agent, TEST_Q)
agent_time   = time.perf_counter() - agent_start

agent_tokens = sum(
    getattr(r.usage, "input_tokens", 0) + getattr(r.usage, "output_tokens", 0)
    for r in agent_result.raw_responses if r.usage
)

print(f"Latency : {agent_time:.2f}s")
print(f"Tokens  : {agent_tokens}")
print(f"\nFinal Answer:\n{agent_result.final_output}")


# ═══════════════════════════════════════════════════════════════════
# METRICS SUMMARY
# ═══════════════════════════════════════════════════════════════════

total_tokens  = dspy_tokens + agent_tokens
total_latency = dspy_time + agent_time

print("\n" + "=" * 60)
print("METRICS SUMMARY")
print("=" * 60)
print(f"{'Stage':<28} {'Latency':>10} {'Tokens':>10}")
print("-" * 50)
print(f"{'Phase 1: DSPy (prompt optimize)':<28} {dspy_time:>9.2f}s {dspy_tokens:>10}")
print(f"{'Phase 2: Agent SDK (execute)':<28} {agent_time:>9.2f}s {agent_tokens:>10}")
print("-" * 50)
print(f"{'TOTAL':<28} {total_latency:>9.2f}s {total_tokens:>10}")
print("=" * 60)
print()
print("Pipeline: Question → DSPy optimizes prompt → Agent executes → Answer")
print("Each tool does only what it is good at. Zero redundancy.")

PHASE 1: DSPy generating optimized prompt + reasoning
Latency : 5.63s
Tokens  : 419

DSPy Reasoning (will become Agent's instructions):
Caffeine, the primary active ingredient in coffee, is a stimulant that affects the central nervous system. When consumed, it blocks the action of adenosine, a brain chemical involved in sleep regulation, leading to increased alertness and wakefulness. Taking coffee before bed can disrupt your sleep patterns, making it harder to fall asleep and reducing the overall quality of sleep. Factors such as the individual's caffeine sensitivity, the amount consumed, and timing can all influence the effects.

DSPy's own answer attempt:
If you drink coffee before going to bed, you are likely to experience difficulty falling asleep and reduced sleep quality. Caffeine can stay in your system for several hours, with a half-life of about 3 to 7 hours, meaning it can keep you alert long after ingestion. For example, if you have a cup of coffee at 8 PM, you may still ha

Latency : 7.85s
Tokens  : 513

Final Answer:
Taking coffee before bed can significantly disrupt your sleep patterns due to the presence of caffeine, a stimulant that affects the central nervous system. Here’s how it works:

1. **Adenosine Blockage**: Caffeine blocks adenosine, a brain chemical that promotes sleepiness. By inhibiting adenosine’s action, caffeine keeps you alert and awake, making it harder to fall asleep.

2. **Increased Alertness**: After consuming coffee, you may feel more energized and alert. This stimulation can last several hours, depending on the amount of coffee consumed and individual sensitivity to caffeine.

3. **Reduced Sleep Quality**: Even if you manage to fall asleep after having coffee, the quality of your sleep may decline. Caffeine can lead to lighter sleep stages and increase the chances of waking up during the night.

4. **Timing and Sensitivity**: The effects of caffeine can vary based on factors such as the amount consumed and the timing relative to 